# 8.0 Actuators and Drive Systems

**ELEC3901 Robotic Manipulators, Lab 8** (week 8: Actuators and Drive Systems)

<br>

Read before this lab: the week 8 lecture and RVC3 Ch. 9.1.6 to 9.1.8 (motors, gearing, friction); supplementary notes. Each section below follows the lecture slides listed next to it.

### Contents

| Section | Where | Time | Lecture slides |
|---|---|---|---|
| [8.1 The servo on paper](#paper) | desk | 5 min | 8, 11-12: The torque-speed line; Why gear; Reflected inertia |
| [8.2 Friction in the twin](#twinfric) | desk | 8 min | 16-17: Friction; Identifying friction |
| [8.3 On the arm: the bus](#bus) | arm | 8 min | 24: The bus |
| [8.4 On the arm: how fast a joint follows](#follow) | arm | 15 min | 9, 25-26: Two time constants; The servo's own trajectory; Streaming your own |
| [8.5 On the arm: friction on joint 1](#armfric) | arm | 15 min | 16-17: Friction; Identifying friction |

**How this notebook works.** The desk sections (8.1 and 8.2) show the lecture's ideas with given code: run it, read it, then
complete the short *Exercises* (a few lines with `...` blanks, checked by `try_it`, which prints `OK` or what looks wrong). The arm
sections (8.3 to 8.5) measure your own servos: the bus, how fast a joint follows a command, and how much friction a joint has. Every
motion moves one joint at a time. Nothing here asks you to write a function. A **Hint** opens when you click it.

*Notation.* $\mathbf{q}$ joint angles (rad), $\dot q$ a joint's speed (rad/s), $\tau$ torque (N m), $N$ the gear ratio, $J_m$ the
motor's rotor inertia. A servo's **load** is its drive as a fraction of the maximum (`arm.read_load()`, from $-1$ to $1$; Lab 7
showed it is not a torque). Equations are numbered (1), (2) through the notebook; slide numbers are the week 8 lecture's.

**Checkpoint for sign-off:** The printed checkpoint line: read and loop times, joint 4's step rise time and overshoot, its amplitude and lag at 1 Hz, and joint 1's Coulomb and viscous friction

This lab needs only the toolbox and the provided modules (`scripts/`); the earlier labs' functions are still imported from `omx_ref.py` if you want them.

> **Safety.** Keep the workspace clear of hands and cables before enabling torque, and set a conservative profile velocity
> (`arm.set_profile(60, 20)` or lower) before commanding motion. If anything unexpected happens, press the power isolator on the
> bench first, then think. Torque off (`arm.torque(False)`) before touching the arm.

The imports. `omx_dyn()` is the toolbox model of the OMX-F with ROBOTIS's masses (Lab 7); `OMXSim` is the digital twin, used at the desk. `XL430` holds the datasheet figures of the servos on joints 1 to 3 (ROBOTIS e-manual). Section 8.3 connects to the arm.

In [ ]:
import sys, os
# Repository layout: weekNN/ (notebooks), scripts/ (helper modules), data/ (your measurements, shared by every week).
for _up in ('.', '..', os.path.join('..', '..')):
    if os.path.isdir(os.path.join(_up, 'scripts')):
        _root = os.path.abspath(_up); sys.path.insert(0, os.path.join(_root, 'scripts'))
        os.makedirs(os.path.join(_root, 'data'), exist_ok=True); os.chdir(os.path.join(_root, 'data'))   # files save/load here
        break
import numpy as np, matplotlib.pyplot as plt, time, cv2
import roboticstoolbox as rtb
from spatialmath import SE3, SO3, SE2, UnitQuaternion
from spatialmath.base import trplot, tranimate
from omx_model import omx_robot, omx_ik, Q_LIM, T_EC, P_TIP_E, mount_tilt, look_down_pitch
from omx import OMX
from omx_camera import Camera, tip_in_mat, predicted_T_CM, pose_error
from omx_calib import small_T
from labcheck import check, try_it
NOM = dict(d1=0.0975, l2z=0.11315, l2x=0.0415, l3=0.162, l4=0.123)   # nominal link parameters (m)
from omx_ref import R_to_rpy, camera_ik, chain_T_BM, condition_number, dh_table, error_stats, fit_gravity, fk_dh, gravity_regressor, gravity_torque, inv_T, jacobian_numeric, make_T, manipulability, mount_from_views, move_metrics, omx_fk, omx_ik, planar_2r, plot_error_field, revolute_jacobian, rot2, rot_about, rotx, roty, rotz, rpy_to_R, se2, static_torque, step_log, straight_line_joints, tracking_rms, transform_point, transform_points, transl2, trot2   # reference versions of PREVIOUS labs' tasks; your own definitions below override them
np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
omx = omx_robot()              # Robotics Toolbox model (nominal)
T_EC_CAD = T_EC   # the design camera mount (ROBOTIS CAD: optical axis 38 deg below the tool). Each arm's camera sits a little differently: Lab 2
                  # measures yours, and from Lab 3 on the MOUNT cell re-measures it every session and replaces T_EC
HOVER = 0.09      # tip hover height with the tool vertical: the OMX-F reaches r ~ 0.24 m there (touching the mat: r ~ 0.28 m)
CAM_HOVER = 0.15  # tip height for camera measurements, tool pitched with look_down_pitch(T_EC) so the camera looks straight down:
                  # the camera is then ~180 mm up, enough to see the marker ring around the clear zone

In [ ]:
from omx_model import omx_dyn, HOME_Q, ARMATURE
from omx_mujoco import OMXSim
import json
robot = omx_dyn()                                    # the OMX with masses (Lab 7)
twin = OMXSim(view=False, realtime=False)            # the digital twin, for the desk sections (no window, simulated time)
twin.torque(True)
twin.set_profile(40, 10)
twin.home(wait=True)
# the XL430-W250-T at the joint, from the ROBOTIS e-manual: stall torque (N m) and no-load speed (rpm) at three supply voltages
XL430 = {'N': 258.5, 'V': [9.0, 11.1, 12.0], 'stall_Nm': [1.0, 1.4, 1.5], 'no_load_rpm': [47, 57, 61]}
RAW_SPEED = 0.229 * 360 / 60                         # set_profile's speed unit: 0.229 rpm = 1.374 deg/s
print('the OMX-F runs its servos from 12 V; the twin uses the 11.1 V stall torque, 1.4 N m')

<br>

<a id='paper'></a>

### 8.1 The servo on paper (desk)
---

In steady state a DC motor's torque falls in a straight line with speed (slide 8), from the **stall torque** $\tau_{\mathrm{stall}}$
at rest to nothing at the **no-load speed** $\dot q_0$:

$$ \tau = \tau_{\mathrm{stall}} \left( 1 - \frac{\dot q}{\dot q_0} \right) \qquad (1) $$

A lower supply voltage moves the whole line down. The XL430's datasheet gives both ends at three voltages, already at the joint
(after the 258.5 : 1 gearbox).

In [ ]:
# --- worked case: the XL430's torque-speed lines, and its most power ---------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 3.4))
for V, stall, rpm in zip(XL430['V'], XL430['stall_Nm'], XL430['no_load_rpm']):
    qd0 = rpm * 2 * np.pi / 60                       # no-load speed (rad/s)
    qd = np.linspace(0, qd0, 50)
    ax.plot(qd, stall * (1 - qd / qd0), label=f'{V:g} V')
ax.set_xlabel('joint speed (rad/s)')
ax.set_ylabel('joint torque (N m)')
ax.grid(alpha=0.3)
ax.legend()
plt.show()
qd0_12 = XL430['no_load_rpm'][2] * 2 * np.pi / 60
print(f'at 12 V: most power at half speed and half torque, {XL430["stall_Nm"][2] * qd0_12 / 4:.1f} W at the joint')

Through a gearbox of ratio $N$ the joint feels the motor's rotor $N^2$ times bigger (slide 12):

$$ J_{\mathrm{eff}} = J_{\mathrm{link}} + N^2 J_m \qquad (2) $$

The model gives every joint the same motor term, `ARMATURE` = 0.01 kg m² at the joint (ROBOTIS publish no rotor inertia, so this
is the digital twin's figure). It is already inside `robot.inertia(q)`.

In [ ]:
# --- Exercise 8.1a: how much of joint 2's inertia is the motor, reaching out level? ----------------------------------
q_level = omx_ik([0.25, 0.0, 0.12], 0.0)             # tool horizontal, 250 mm out
M_level = ...                      # the inertia matrix at q_level, from the toolbox
share = ...                        # the motor's share of joint 2's inertia M[1, 1]: ARMATURE divided by it
J_m = ...                          # the rotor's own inertia (kg m^2): equation (2), N^2 J_m = ARMATURE
try_it("the motor's share of joint 2's inertia", share, ARMATURE / twin.inertia(q_level)[1, 1], tol=1e-6)
try_it('rotor inertia J_m (kg m^2)', J_m, 0.01 / 258.5 ** 2, tol=1e-12);

<details><summary><b>Hint</b></summary>

`robot.inertia(q)` returns $\mathbf{M}(\mathbf{q})$, 4 x 4; joint 2's entry is `[1, 1]` (Python counts from 0). The share is
`ARMATURE / M_level[1, 1]`, and (2) gives $J_m$ = `ARMATURE / XL430['N'] ** 2`.

</details>

<br>

<a id='twinfric'></a>

### 8.2 Friction in the twin (desk)
---

The lecture's friction model, Coulomb plus viscous (slide 16):

$$ \tau_f = \tau_c\,\mathrm{sgn}(\dot q) + b\,\dot q \qquad (3) $$

To measure it, turn a joint at a **constant speed**: no acceleration, so nothing but friction (and gravity) needs torque. Joint 1
turns about a vertical axis, so gravity never loads it: its load at constant speed is all friction (slide 17). The given function
below does that. `set_profile(V, 0)` sets the speed limit and no acceleration ramp, so `move_q` runs the whole move at one speed;
while it moves, the loop reads the speed and the load, keeping only the middle 60° of the sweep.

In [ ]:
# --- given: constant-speed runs of joint 1, both ways, at several speeds --------------------------------------------------
def sweep(arm, speeds_deg, half_span_deg=60, middle_deg=30, timeout=30):
    # returns the mean speed (rad/s) and the mean load (fraction) of joint 1 over the middle of each run, both directions
    home = np.array(HOME_Q, dtype=float)
    mean_speed = []
    mean_load = []
    for speed in speeds_deg:
        for direction in (1, -1):
            start = home.copy()
            start[0] = -direction * np.radians(half_span_deg)
            goal = home.copy()
            goal[0] = direction * np.radians(half_span_deg)
            arm.set_profile(40, 10)                  # a gentle move to the start
            arm.move_q(start, wait=True)
            arm.set_profile(max(1, int(round(speed / RAW_SPEED))), 0)    # this speed, no ramp: a constant-speed move
            arm.move_q(goal)
            speeds = []
            loads = []
            t_end = time.time() + timeout
            while time.time() < t_end:
                q = arm.read_q()
                if abs(q[0]) < np.radians(middle_deg):               # the middle of the sweep only
                    speeds.append(arm.read_qd()[0])
                    loads.append(arm.read_load()[0])
                if abs(q[0] - goal[0]) < np.radians(1):              # arrived
                    break
            mean_speed.append(np.mean(speeds))
            mean_load.append(np.mean(loads))
    arm.set_profile(40, 10)
    arm.move_q(home, wait=True)
    return np.array(mean_speed), np.array(mean_load)

SPEEDS = [10, 20, 40, 60]                            # deg/s
v_twin, load_twin = sweep(twin, SPEEDS)
for v, load in zip(v_twin, load_twin):
    print(f'speed {np.degrees(v):6.1f} deg/s   load {100 * load:5.2f} %')

Equation (3) is linear in $\tau_c$ and $b$: one column of $\mathrm{sgn}(\dot q)$ and one of $\dot q$, then least squares, exactly as
Lab 7 fitted the masses. Here in load units; the twin was built with $\tau_c$ = 0.1 N m and $b$ = 0.05 N m s/rad, and its loads are
torques divided by its stall torque, 1.4 N m.

In [ ]:
# --- Exercise 8.2a: fit Coulomb and viscous friction to the twin's runs -------------------------------------------------
A = ...                            # two columns: sgn(speed) and speed (use np.column_stack and np.sign)
fit_twin = ...                     # least squares: np.linalg.lstsq(A, load_twin, rcond=None)[0], two numbers
tc_twin = ...                      # the first: Coulomb friction
b_twin = ...                       # the second: viscous friction
try_it('Coulomb friction (load fraction)', tc_twin, 0.1 / 1.4, tol=0.003)
try_it('viscous friction (load fraction per rad/s)', b_twin, 0.05 / 1.4, tol=0.003);

<details><summary><b>Hint</b></summary>

`np.column_stack([np.sign(v_twin), v_twin])` makes the two columns. `lstsq` returns several things; `[0]` is the solution, two
numbers: `fit_twin[0]` and `fit_twin[1]`.

</details>

<br>

<a id='bus'></a>

### 8.3 On the arm: the bus (arm)
---

From here on you need the arm, or the digital twin (`SIMULATE = True` in the next cell; the twin has no bus, so its times are your
computer's). The camera is not used. Every motion in this lab moves **one joint** at home or near it: keep the space around the arm
clear, and keep a hand near the power switch.

All six servos share one wire at 1 Mbaud (slide 24). `arm.read_q()` asks all four joints in one **sync read** packet;
`arm.read_load()` asks each servo in turn. Time both, and a loop that writes a goal and reads the joints back, as a stream does.

In [ ]:
# --- connect to the arm --------------------------------------------------------------------------------------------
# Safe to re-run: an existing connection is closed first, so the COM port is released before it is opened again.
# SIMULATE = True runs the MuJoCo digital twin (omx_mujoco.py) instead of the real arm: same API, a rendered
# wrist camera, and optional hidden model errors so you can test calibration against ground truth.
SIMULATE = False
PORT = 'auto'                  # finds the OpenRB-150 by its USB id; or set it, e.g. 'COM4' (Device Manager) / '/dev/ttyACM0'
# --- which arm are you on today? Read the asset label on the base. Everything you measure is tagged with it,
#     because zero offsets, camera intrinsics and the camera mount are all properties of THIS arm.
ARM_ID = 'OMX-00'            # EDIT every session
if 'arm' in globals():
    try:
        arm.close()
    except Exception as e:
        print('closing the previous connection:', e)
    del arm
if SIMULATE:
    from omx_mujoco import OMXSim, SimCamera
    arm = OMXSim(view=True)    # view=False if no display; set kin_error=..., joint_offsets=... to hide errors
    # Video: arm.record('run.mp4') writes the wrist camera and a scene camera side by side (same size) while the arm
    # moves; arm.stop_recording() when done.
else:
    arm = OMX(PORT)
print('connected:', 'digital twin' if SIMULATE else arm.port_name, '| arm', ARM_ID)

In [ ]:
# --- given: time 200 of each, after the arm is at home ----------------------------------------------------------------
arm.torque(True)
arm.set_profile(40, 10)
arm.home(wait=True)
q_home = arm.read_q()

def time_calls(call, n=200):
    # the time each of n calls takes (ms)
    times = []
    for k in range(n):
        t0 = time.perf_counter()
        call()
        times.append(1e3 * (time.perf_counter() - t0))
    return np.array(times)

t_read_q = time_calls(arm.read_q)                                    # one sync read: four joints in one packet
t_read_load = time_calls(arm.read_load)                              # four separate reads, one per servo
t_loop = time_calls(lambda: (arm.move_q(q_home), arm.read_q()))      # write a goal, read back: one streaming step
fig, ax = plt.subplots(figsize=(7, 3))
for times, name in ((t_read_q, 'read_q (sync read)'), (t_read_load, 'read_load (4 reads)'), (t_loop, 'write + read')):
    ax.hist(times, bins=40, alpha=0.6, label=name)
ax.set_xlabel('time per call (ms)')
ax.set_ylabel('calls')
ax.legend()
plt.show()

**Your turn.** A stream's period must cover at least one write-and-read. Find the typical (median) times, the fastest loop rate
your bus allows, and how many times longer Lab 6's 50 ms stream period was than one loop.

In [ ]:
# --- Exercise 8.3a: what the bus allows ---------------------------------------------------------------------------------
ms_read_q = ...                    # the median of t_read_q (ms)
ms_loop = ...                      # the median of t_loop (ms)
rate_max = ...                     # the fastest loop rate (Hz): one loop every ms_loop milliseconds
headroom = ...                     # Lab 6's period, 50 ms, divided by ms_loop
try_it('median read_q (ms)', ms_read_q, float(np.percentile(t_read_q, 50)), tol=1e-9)
try_it('fastest loop rate (Hz)', rate_max, 1000 / float(np.percentile(t_loop, 50)), tol=1e-6)
try_it("Lab 6's period in loops", headroom, 50 / float(np.percentile(t_loop, 50)), tol=1e-6);

<details><summary><b>Hint</b></summary>

`np.median(t_read_q)`. A loop that takes `ms_loop` milliseconds runs `1000 / ms_loop` times a second.

</details>

<br>

<a id='follow'></a>

### 8.4 On the arm: how fast a joint follows (arm)
---

Inside each servo a controller turns a goal into motor drive (slides 9 and 21). Two ways to see how fast it follows: a **step** (a
goal that jumps) and a **sine** (a goal that keeps moving, streamed as in Lab 6, slide 26). Joint 4, the wrist, moves 10° each way
from home.

First the step. `set_profile(300, 0)` makes the servo's own trajectory almost instant (slide 25), so the goal jumps and what you see
is the servo's controller.

In [ ]:
# --- given: a 10 degree step on joint 4, logged as fast as the bus allows -----------------------------------------------
arm.set_profile(40, 10)
arm.move_q(q_home, wait=True)
goal = q_home.copy()
goal[3] = q_home[3] + np.radians(10)
arm.set_profile(300, 0)                              # no ramp, a very short move time: the goal jumps
t_step = []
y_step = []
t0 = time.perf_counter()
arm.move_q(goal)
while time.perf_counter() - t0 < 0.6:
    t_step.append(time.perf_counter() - t0)
    y_step.append(np.degrees(arm.read_q()[3] - q_home[3]))
t_step = np.array(t_step)
y_step = np.array(y_step)
arm.set_profile(40, 10)
arm.move_q(q_home, wait=True)
plt.figure(figsize=(6, 3))
plt.plot(1e3 * t_step, y_step, '.-', ms=3)
plt.axhline(10, color='k', lw=0.8, ls=':')
plt.xlabel('time (ms)')
plt.ylabel('joint 4 - home (deg)')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# --- Exercise 8.4a: rise time and overshoot ------------------------------------------------------------------------------
t10 = ...                          # the first time y_step reaches 1 degree (10 % of the step): t_step[np.argmax(y_step >= 1)]
t90 = ...                          # the first time it reaches 9 degrees (90 %)
rise_ms = ...                      # rise time, 10 % to 90 % (ms)
overshoot = ...                    # how far the largest reading goes past 10 degrees, as a percentage of the step (0 if it never does)
_first = lambda level: float(t_step[np.nonzero(y_step >= level)[0][0]])
try_it('rise time (ms)', rise_ms, 1e3 * (_first(9) - _first(1)), tol=1e-6)
try_it('overshoot (%)', overshoot, max(0.0, 10 * (y_step.max() - 10)), tol=1e-6);

<details><summary><b>Hint</b></summary>

`y_step >= 1` is True from the first reading at or above 1°; `np.argmax` of it gives that reading's index. The overshoot is
`(y_step.max() - 10) / 10 * 100`, or 0 if that is negative (`max(0.0, ...)`).

</details>

Now the sine. Each frequency is streamed for three periods at 50 samples a second (`dt` = 20 ms, which your bus allows: Exercise
8.3a). The speed limit is raised to `set_profile(100, 0)`, 137°/s, so `stream` does not refuse the faster sines (a 10° sine at 2 Hz
peaks at 126°/s). To measure how well the joint follows, fit a sine of the same frequency to what it did, ignoring the first period
while it starts:

$$ y(t) \approx a \sin(2\pi f t) + c \cos(2\pi f t) + d \qquad (4) $$

linear in $a$, $c$ and $d$, so least squares again. Its amplitude is $\sqrt{a^2 + c^2}$ (the goal's is 10°), and it trails the
goal by the phase $\mathrm{atan2}(-c, a)$, a lag of $\mathrm{atan2}(-c, a) / (2\pi f)$ seconds.

In [ ]:
# --- given: stream a 10 degree sine on joint 4 at four frequencies --------------------------------------------------------
FREQS = [0.25, 0.5, 1.0, 2.0]                        # Hz
DT = 0.02                                            # s
runs = []
arm.set_profile(100, 0)                              # 137 deg/s: fast enough for 10 degrees at 2 Hz
for f in FREQS:
    t_goal = np.arange(0, 3 / f, DT)
    qs = np.tile(q_home, (len(t_goal), 1))
    qs[:, 3] = q_home[3] + np.radians(10) * np.sin(2 * np.pi * f * t_goal)
    t_meas, q_meas = arm.stream(qs, DT)
    runs.append((f, t_meas, np.degrees(q_meas[:, 3] - q_home[3])))
    arm.move_q(q_home, wait=True)
arm.set_profile(40, 10)
fig, ax = plt.subplots(1, 4, figsize=(13, 2.8), sharey=True)
for k, (f, t_meas, y) in enumerate(runs):
    ax[k].plot(t_meas, 10 * np.sin(2 * np.pi * f * t_meas), 'k:', lw=1, label='goal')
    ax[k].plot(t_meas, y, label='joint 4')
    ax[k].set_title(f'{f} Hz')
    ax[k].set_xlabel('t (s)')
ax[0].set_ylabel('deg from home')
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- Exercise 8.4b: amplitude and lag at each frequency ------------------------------------------------------------------
amp_ratio = []
lag_ms = []
for f, t_meas, y in runs:
    keep = t_meas >= 1 / f                           # skip the first period
    t = t_meas[keep]
    w = 2 * np.pi * f
    A = ...                        # three columns: sin(w t), cos(w t) and ones, equation (4)
    coef = ...                     # least squares for the readings y[keep]: a, c, d = coef[0], coef[1], coef[2]
    amp_ratio.append(...)          # the fitted amplitude, sqrt(a^2 + c^2), divided by the goal's 10 degrees
    lag_ms.append(...)             # the lag in ms: 1000 * atan2(-c, a) / w
def _fit(f, t_meas, y):
    keep = t_meas >= 1 / f
    w = 2 * np.pi * f
    s = np.linalg.pinv(np.column_stack([np.sin(w * t_meas[keep]), np.cos(w * t_meas[keep]), np.ones(keep.sum())])) @ y[keep]
    return np.hypot(s[0], s[1]) / 10, 1e3 * np.arctan2(-s[1], s[0]) / w
try_it('amplitude ratio at each frequency', amp_ratio, [_fit(*r)[0] for r in runs], tol=1e-6)
try_it('lag at each frequency (ms)', lag_ms, [_fit(*r)[1] for r in runs], tol=1e-6);

<details><summary><b>Hint</b></summary>

`np.column_stack([np.sin(w * t), np.cos(w * t), np.ones(len(t))])`, then `np.linalg.lstsq(A, y[keep], rcond=None)[0]` gives three
numbers, `coef`. With `a = coef[0]` and `c = coef[1]`, the amplitude is `np.hypot(a, c)`. A stream's own samples are `DT` apart, and each is a move of `DT`, so expect at least
one period of lag (20 ms) on top of what the servo adds.

</details>

<br>

<a id='armfric'></a>

### 8.5 On the arm: friction on joint 1 (arm)
---

The twin's friction is exactly (3). Your arm's is whatever its gearbox does. Run Section 8.2's `sweep` on the arm: joint 1 turns
from -60° to +60° and back at 10, 20, 40 and 60°/s, about a minute in all, at home pose. **Keep the space around the arm clear.**

In [ ]:
# --- given: the same constant-speed runs on your arm's joint 1 ------------------------------------------------------------
v_arm, load_arm = sweep(arm, SPEEDS)
for v, load in zip(v_arm, load_arm):
    print(f'speed {np.degrees(v):6.1f} deg/s   load {100 * load:5.2f} %')

In [ ]:
# --- Exercise 8.5a: your arm's friction, and how much of it is viscous ---------------------------------------------------
A_arm = ...                        # the two columns of (3) for your runs
fit_arm = ...                      # least squares on load_arm
tc_arm = ...                       # Coulomb friction: the first number
b_arm = ...                        # viscous friction: the second
v_fast = np.radians(60)            # the fastest run (rad/s)
viscous_share = ...                # the viscous part's share of the friction at v_fast: b v / (tc + b v)
_s = lambda: np.linalg.pinv(np.column_stack([np.sign(v_arm), v_arm])) @ load_arm
try_it('Coulomb friction (load fraction)', tc_arm, _s()[0], tol=1e-9)
try_it('viscous friction (load fraction per rad/s)', b_arm, _s()[1], tol=1e-9)
try_it('viscous share of the friction at 60 deg/s', viscous_share, _s()[1] * np.radians(60) / (_s()[0] + _s()[1] * np.radians(60)), tol=1e-9);

<details><summary><b>Hint</b></summary>

As in Exercise 8.2a with `v_arm` and `load_arm`. At speed $\dot q$ the friction is $\tau_c + b\,\dot q$ (load units), and the
viscous part is $b\,\dot q$ of it. A share near 0 means the joint's friction is almost all Coulomb: nearly the same at any speed.

</details>

In [ ]:
# your arm against the twin, and the checkpoint (after the exercises)
_values = [tc_twin, b_twin, ms_read_q, ms_loop, rise_ms, overshoot, tc_arm, b_arm] + list(amp_ratio) + list(lag_ms)
_done = True
for _v in _values:
    if _v is ...:
        _done = False
if not _done:
    print('finish Exercises 8.2a to 8.5a first: then this cell draws the comparison and prints the checkpoint')
else:
    v_line = np.linspace(-1.1, 1.1, 200)
    v_line = v_line[v_line != 0]
    plt.figure(figsize=(6, 3.4))
    plt.plot(np.degrees(v_twin), 100 * load_twin, 's', label='twin')
    plt.plot(np.degrees(v_line), 100 * (tc_twin * np.sign(v_line) + b_twin * v_line), ':', color='C0')
    plt.plot(np.degrees(v_arm), 100 * load_arm, 'o', label=f'arm {ARM_ID}')
    plt.plot(np.degrees(v_line), 100 * (tc_arm * np.sign(v_line) + b_arm * v_line), '-', color='C1')
    plt.axhline(0, color='k', lw=0.5)
    plt.axvline(0, color='k', lw=0.5)
    plt.xlabel('joint 1 speed (deg/s)')
    plt.ylabel('load (%)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
    k1 = FREQS.index(1.0)
    json.dump({'arm_id': ARM_ID, 'read_q_ms': float(ms_read_q), 'loop_ms': float(ms_loop), 'step_rise_ms': float(rise_ms),
               'step_overshoot_pct': float(overshoot), 'freqs_hz': FREQS, 'amp_ratio': [float(x) for x in amp_ratio],
               'lag_ms': [float(x) for x in lag_ms], 'joint1_coulomb_load': float(tc_arm), 'joint1_viscous_load_per_rad_s': float(b_arm)},
              open('lab8_actuators.json', 'w'), indent=1)
    # --- checkpoint: show this line to a demonstrator ------------------------------------------------------------------
    print(f'ARM {ARM_ID} | read_q {ms_read_q:.2f} ms, loop {ms_loop:.2f} ms | step rise {rise_ms:.0f} ms, overshoot {overshoot:.1f} % '
          f'| 1 Hz: amplitude {amp_ratio[k1]:.2f}, lag {lag_ms[k1]:.0f} ms | joint 1 friction {100 * tc_arm:.1f} % + {100 * b_arm:.1f} % per rad/s '
          f'| saved lab8_actuators.json')
    arm.home(wait=True)

**The calls in this lab:** `inertia(q)` from the toolbox; on the arm `read_q()`, `read_qd()`, `read_load()`, `move_q(q)`,
`set_profile(V, A)` (with `A = 0`, a constant-speed move) and `stream(qs, dt)`; and `np.linalg.lstsq` for every fit.

## Before you leave
1. Park the arm (`arm.rest()`: the folded rest pose, where it stays put with torque off), then `arm.torque(False)` and `arm.close()`.
   Never switch torque off at home or mid-task: the arm drops under gravity.
2. Switch off the bench power supply and tidy cables.
3. Show your checkpoint output to a demonstrator for sign-off.
4. Commit this notebook and any data files to your own LABS repository and **push** (Sync Changes): later labs read them. The assessment repository (issued in week 6) starts fresh; copy across any code you want to reuse.

In [ ]:
arm.set_profile(40, 10); arm.rest(wait=True)     # folded rest pose: stable with torque off
arm.torque(False)
arm.close()
if globals().get('cam') is not None: cam.close()